<a href="https://colab.research.google.com/github/yukien462/mysite/blob/master/%E6%96%99%E9%87%91%E7%A2%BA%E8%AA%8D%E7%A5%A8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## はじめに
- 初めに以下のセルを実行してください
- 必要なライブラリのインポートとデータベース（PostgreSQL）からのデータ読み込みを行います
- pandas等、利用が想定されるライブラリは以下セルでインポートしています
- その他利用したいライブラリがあれば適宜インストールしてください（"!pip install ライブラリ名"でインストールも可能）
- 処理は複数回に分けても構いません

In [ ]:
from google.colab import drive
drive.mount('/content/drive/')

Drive already mounted at /content/drive/; to attempt to forcibly remount, call drive.mount("/content/drive/", force_remount=True).


In [ ]:
import os
import pandas as pd
import numpy as np

In [ ]:
!pip install chardet

### PDF変換用のツール（LibreOffice）のインストール
Google Colab (Linux環境) でExcelファイルをPDFに変換するため、ヘッドレス（画面なし）で動作する LibreOffice をインストールします。

In [ ]:
# LibreOfficeのインストール（Colab環境にPDF変換エンジンを導入します）
#!apt-get update
#!apt-get install -y libreoffice

### 日本語フォントのインストール
LibreOfficeで日本語が文字化けするのを防ぐため、システムに日本語フォントをインストールします。

In [ ]:
# 日本語フォントをインストールし、キャッシュを更新します
#!apt-get install -y fonts-noto-cjk fonts-ipafont-gothic fonts-ipafont-mincho
#!fc-cache -fv

In [ ]:
#  文字コードを調べる
import chardet

with open('/content/drive/MyDrive/請求書発行/Data.csv','rb') as f:
  data = f.read()
  result = chardet.detect(data)
  print(result)

{'encoding': 'SHIFT_JIS', 'confidence': 0.99, 'language': 'Japanese'}


In [ ]:
"""
データ読み込み
"""
import pandas as pd

def Read_CSV():
  # CSVファイルの最初の3行をテキストとして読み込む（エラーを避けるため）
  with open('/content/drive/MyDrive/請求書発行/Data.csv', mode='r', encoding='SHIFT_JIS') as f:
      # 最初の3行をそれぞれ取得して、前後の空白や改行を削除したリストを作る
      head_lines = [f.readline().strip().strip("'\"") for _ in range(3)] # i in range(3) のiを使わないため"_"を表記
  #head_data = pd.DataFrame({"ヘッダー情報": head_lines})
  # 4行目以降（実データ）をデータフレームとして読み込む
  df = pd.read_csv('/content/drive/MyDrive/請求書発行/Data.csv', skiprows=3, encoding='SHIFT_JIS')
  return df
  #display(df)
  #print(head_lines[0])
  #print(df['利用者'][0])

In [ ]:
"""
料金確認票を作成
"""
def Create_BillState(df):
  # 研究室ごとに
  df_by_LaboCode = df.groupby('利用者の研究室コード').agg({'利用料金（小計）':'sum'}).reset_index()
  display(df_by_LaboCode)

  df_contents = df[['利用者の研究室コード','利用設備','利用者の研究室','利用者','利用料金（単価）','利用時間','利用料金（単位数）','利用料金（小計）','研究室費、私費']]
  #display(df_contents[df_contents['利用者の研究室コード'] =='BIOSF'])

  return df_contents

### PDF出力関数の実装
LibreOfficeのコマンドを使って、保存したExcelファイルをPDFに変換する関数 `Print_PDF` の実装例です。　AIさんが作成

In [ ]:

import subprocess
import os
import openpyxl
from openpyxl import utils
from copy import copy

def Print_PDF(df):
    """
    データフレームを元にExcelファイルを作成し、それをPDFに変換して保存します。
    """
    src = 10   # 10 行目をコピー
    dest = 10  # 10 行目に挿入
    # Excelテンプレートのパス
    template_path = '/content/drive/MyDrive/請求書発行/template.xlsx'
    output_dir = '/content/drive/MyDrive/請求書発行/'

    # 研究室コードごとにデータを処理する例
    grouped = df.groupby('利用者の研究室コード')

    for lab_code, group in grouped:

        # Excelテンプレートを開く
        wb = openpyxl.load_workbook(template_path)
        ws = wb.active

        # 例: 最初の行の情報をヘッダーとして書き込む
        first_row = group.iloc[0]
        ws['B3'] = f"{first_row['利用者の研究室']} 御中" # 宛名など
        ws['B5'] = f"{first_row['利用設備']}"

        #  利用データ
        labo_equip_count = len(group) # 件数
        ws.insert_rows(dest)
        for col in range(1, labo_equip_count -1):
          src_cell = ws.cell(row=10, column=)



        # 一時的なExcelファイル名を作成
        temp_xlsx_name = f"請求書_{first_row['利用者の研究室']}.xlsx"
        temp_xlsx_path = os.path.join(output_dir, temp_xlsx_name)

        # Excelとして一旦保存
        wb.save(temp_xlsx_path)
        wb.close()

        print(f"Excel保存完了: {temp_xlsx_name}")

        # LibreOfficeを使ってExcelをPDFに変換 (コマンドライン実行)
        # --headless: 画面を起動しないモード
        # --convert-to pdf: PDFに変換
        # --outdir: 出力先ディレクトリ
        cmd = [
            "libreoffice",
            "--headless",
            "--convert-to", "pdf",
            "--outdir", output_dir,
            temp_xlsx_path
        ]

        try:
            # コマンドの実行
            subprocess.run(cmd, check=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
            print(f"PDF変換完了: 請求書_{first_row['利用者の研究室']}.pdf")

            # 必要に応じて、中間生成物である一時Excelファイルを削除します
            # os.remove(temp_xlsx_path)
        except subprocess.CalledProcessError as e:
            print(f"PDF変換エラー ({first_row['利用者の研究室']}): {e.stderr.decode('utf-8')}")

In [ ]:
"""
1列だけをコピーする
"""
def CopyCol(ws,src,dest):
	for i in range(1, ws.max_row+1):
		value = ws.cell(row=i, column=src).value
		#セルの値が数式の場合
		if str(value)[0] == "=":
			#コピー元と同じ列が数式に使われているとき、コピー先の列に変更
			value = value.replace(utils.cell.get_column_letter(src),utils.cell.get_column_letter(dest))
			#コピー元の左の列を使われているとき、コピー先の左の列に変更
			value = value.replace(utils.cell.get_column_letter(src-1),utils.cell.get_column_letter(dest-1))
		#セルの値をコピー
		ws.cell(row=i, column=dest).value = value
		#セルのスタイルをコピー
		ws.cell(row=i, column=dest)._style = copy(ws.cell(row=i, column=src)._style)

In [ ]:
"""
行をcount分を挿入する
"""
def CopyRow(ws,src,dest,row_count):
	ws.insert_rows(dest, amount=row_count)
	for col in range(dest, dest + count): # スタイルのコピー
			#"@@@@@@@@@@@@@@@@@@@@@@"



In [ ]:
def main():
  df = Read_CSV()           # CSV読み取り
  df = Create_BillState(df) # データ加工
  Print_PDF(df)             # PDF出力

In [ ]:
if __name__=="__main__":
  main()

,利用者の研究室コード,利用料金（小計）
0,BIOSF,5400


Excel保存完了: 請求書_バイオマス構造機能学研究室（堀川）.xlsx
PDF変換完了: 請求書_バイオマス構造機能学研究室（堀川）.pdf
